# Week 2 Day 4 — Yield Curve Conversions
Tests for `zero_to_discount`, `discount_to_zero`, `zeros_to_par`, `par_to_zeros`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.termstructure.bonds.conversions import (
    zero_to_discount,
    discount_to_zero,
    zeros_to_par,
    par_to_zeros,
)

## 1. Zero rate ↔ Discount factor

A 5% zero rate for 10 years (semi-annual) should give a discount factor < 1,
and the inverse should recover exactly 5%.

In [ ]:
z = 0.05
T = 10.0

df = zero_to_discount(z, T)
z_back = discount_to_zero(df, T)

print(f'Zero rate:              {z:.6f}')
print(f'Discount factor:        {df:.6f}')
print(f'Recovered zero rate:    {z_back:.6f}')
print(f'Round-trip error:       {abs(z - z_back):.2e}')

### Interpretation
A discount factor of ~0.6139 means $1 received in 10 years is worth about 61 cents today at a 5% rate. The round-trip error should be at floating-point machine precision (~1e-16).

In [ ]:
# Sweep across maturities to see how discount factors decay
mats = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])
rates_flat = np.full(len(mats), 0.04)  # flat 4% curve

dfs = zero_to_discount(rates_flat, mats)

plt.figure(figsize=(8, 4))
plt.plot(mats, dfs, 'o-', color='steelblue')
plt.xlabel('Maturity (years)')
plt.ylabel('Discount factor')
plt.title('Discount factors on a flat 4% zero curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Flat curve sanity check

On a flat zero curve, par yields must equal the zero rate at every maturity.
This is the single most important sanity check for `zeros_to_par`.

In [ ]:
mats_semi = np.arange(0.5, 10.5, 0.5)  # 0.5, 1.0, ..., 10.0
zeros_flat = np.full(len(mats_semi), 0.04)

pars_flat = zeros_to_par(mats_semi, zeros_flat)

max_err = np.max(np.abs(pars_flat - 0.04))
print(f'Max deviation from 4% on flat curve: {max_err:.2e}')
print(f'First par yield: {pars_flat[0]:.8f}')
print(f'Last  par yield: {pars_flat[-1]:.8f}')

## 3. Upward-sloping curve: par yield < zero rate

When the zero curve slopes upward, par yields should lie **below** zero rates.
Why? Early coupons get discounted at lower rates (cheap), pulling the effective
yield down relative to the final maturity's spot rate.

In [ ]:
zeros_up = 0.02 + 0.003 * mats_semi   # 2% at 0.5Y rising to ~5% at 10Y

pars_up = zeros_to_par(mats_semi, zeros_up)

plt.figure(figsize=(8, 4))
plt.plot(mats_semi, zeros_up * 100, label='Zero rates', color='steelblue')
plt.plot(mats_semi, pars_up * 100, label='Par yields', color='coral', linestyle='--')
plt.xlabel('Maturity (years)')
plt.ylabel('Rate (%)')
plt.title('Zero rates vs. par yields — upward-sloping curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Par yield < zero rate at every point?', np.all(pars_up < zeros_up))

## 4. Round-trip: zeros → par → zeros

`par_to_zeros` bootstraps zero rates back from the par curve.
The recovered zeros should match the originals to machine precision.

In [ ]:
zeros_recovered = par_to_zeros(mats_semi, pars_up)

max_err = np.max(np.abs(zeros_recovered - zeros_up))
print(f'Max round-trip error: {max_err:.2e}')

plt.figure(figsize=(8, 4))
plt.plot(mats_semi, zeros_up * 100, label='Original zeros', color='steelblue', linewidth=2)
plt.plot(mats_semi, zeros_recovered * 100, label='Bootstrapped zeros', color='coral',
         linestyle='--', linewidth=1.5)
plt.xlabel('Maturity (years)')
plt.ylabel('Rate (%)')
plt.title('Round-trip: zeros → par → zeros (lines should overlap exactly)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Round-trip on a realistic inverted curve

Inverted curves (short rates > long rates) occurred in 2022–2023.
Bootstrapping must still work — no special-casing.

In [ ]:
# Inverted: starts high, falls off
zeros_inv = 0.055 - 0.002 * mats_semi   # 5.5% at 0.5Y down to ~3.5% at 10Y

pars_inv = zeros_to_par(mats_semi, zeros_inv)
zeros_inv_back = par_to_zeros(mats_semi, pars_inv)

max_err = np.max(np.abs(zeros_inv_back - zeros_inv))
print(f'Max round-trip error (inverted curve): {max_err:.2e}')
print('Par yield > zero rate at every point?', np.all(pars_inv > zeros_inv))

On an inverted curve, par yields lie **above** zero rates — the mirror image of the upward-sloping case.

## 6. Discount factor properties

Quick checks that discount factors obey basic math:
- Must be in (0, 1) for positive rates
- Must be monotonically decreasing in maturity
- At T=0, df should equal 1

In [ ]:
dfs_check = zero_to_discount(zeros_up, mats_semi)

print('All dfs in (0, 1)?       ', np.all((dfs_check > 0) & (dfs_check < 1)))
print('Monotonically decreasing?', np.all(np.diff(dfs_check) < 0))
print('df at T→0 (0.5Y first):  ', dfs_check[0])  # close to 1 for short mat
print('df at T=0 exactly:       ', zero_to_discount(0.04, 0.0))  # edge case: T=0